# Automated date-time features with Feature-engine

This notebook demonstrates Feature-engine's DatetimeFeatures transformer. It automates common pandas datetime extraction and can be placed in a scikit-learn Pipeline so the same recipe is used during training, validation, and prediction.

## Why use a transformer?

Manual pandas code is useful for exploration, but a reusable transformer is safer in a model workflow. It keeps the selected feature list, original-column policy, parsing options, and transformation order together.

In [ ]:
import importlib.util,subprocess,sys
required={'numpy':'numpy','pandas':'pandas','matplotlib':'matplotlib','seaborn':'seaborn','sklearn':'scikit-learn','feature_engine':'feature-engine'}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from feature_engine.datetime import DatetimeFeatures
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,r2_score
sns.set_theme(style='whitegrid',context='notebook')

## 1. Create a reproducible event dataset

The target is synthetic demand. The model should learn that demand changes by hour and weekday. In a real project, replace this dataframe with a table containing an event timestamp and a target.

In [ ]:
rng=np.random.default_rng(21)
raw=pd.DataFrame({'event_time':pd.date_range('2023-01-01',periods=24*90,freq='h')})
hour=raw['event_time'].dt.hour.to_numpy()
weekday=raw['event_time'].dt.dayofweek.to_numpy()
raw['demand']=50+15*np.sin(2*np.pi*hour/24)+8*(weekday<5)+rng.normal(0,3,len(raw))
raw.head()

## 2. Configure DatetimeFeatures

We request calendar and clock components and drop the raw datetime after extraction. Setting utc=True gives a consistent parsing policy. Choose the feature list deliberately; extracting every possible field can add noise and redundancy.

In [ ]:
features=['year','quarter','month','week','day_of_week','day_of_month','day_of_year','weekend','month_start','month_end','hour','minute','second']
datetime_step=DatetimeFeatures(variables=['event_time'],features_to_extract=features,drop_original=True,missing_values='ignore',utc=True)
X_features=datetime_step.fit_transform(raw[['event_time']])
print('Created columns:',X_features.columns.tolist())
display(X_features.head())

## 3. Put the transformer and model in a pipeline

This is a chronological example: earlier rows train the model and later rows test it. A random split may be acceptable for some tabular tasks, but future prediction should respect time order.

In [ ]:
split=int(len(raw)*.80)
X=raw[['event_time']].copy(); y=raw['demand'].copy()
X_train,X_test=X.iloc[:split],X.iloc[split:]
y_train,y_test=y.iloc[:split],y.iloc[split:]
date_pipeline=Pipeline([
    ('datetime',DatetimeFeatures(variables=['event_time'],features_to_extract=features,drop_original=True,missing_values='ignore',utc=True)),
    ('model',RandomForestRegressor(n_estimators=150,min_samples_leaf=3,random_state=42,n_jobs=-1))
])
date_pipeline.fit(X_train,y_train)
predictions=date_pipeline.predict(X_test)
print('MAE:',round(mean_absolute_error(y_test,predictions),3))
print('R2:',round(r2_score(y_test,predictions),3))

## 4. Inspect predictions

The plot compares actual and predicted demand over the held-out future period. A good score is not enough: check whether errors grow at particular hours, weekdays, or seasonal periods.

In [ ]:
prediction_view=pd.DataFrame({'time':X_test['event_time'].to_numpy(),'actual':y_test.to_numpy(),'predicted':predictions})
prediction_view['residual']=prediction_view['actual']-prediction_view['predicted']
fig,axes=plt.subplots(1,2,figsize=(15,4))
sns.lineplot(data=prediction_view,x='time',y='actual',label='actual',ax=axes[0])
sns.lineplot(data=prediction_view,x='time',y='predicted',label='predicted',ax=axes[0]); axes[0].set_title('Future-period demand')
sns.scatterplot(data=prediction_view,x='predicted',y='residual',alpha=.6,ax=axes[1]); axes[1].axhline(0,color='black',linestyle='--'); axes[1].set_title('Residuals')
plt.tight_layout(); plt.show()

## Practical notes

- Current Feature-engine supports calendar and clock features such as month, quarter, year, week, day of week, day of month, day of year, weekend, period boundaries, leap-year information, hour, minute, and second.
- Parse ambiguous dates with dayfirst, yearfirst, format, or an explicit source contract.
- Keep timezone policy consistent between fit and transform.
- Do not use future timestamps or completion dates that are unavailable at prediction time.
- Consider cyclic sine/cosine features for repeating hour, weekday, or month patterns.

## Summary

DatetimeFeatures makes common date-time extraction repeatable and pipeline-friendly. It is a convenient automation layer over pandas datetime operations, not a replacement for deciding which temporal signals are meaningful and available at prediction time.